# Member 3 Computational Experiment

This notebook runs the Member 3 workflow for baseline comparison, optimized heuristic routing, visualization, and robustness testing.

## 1. Imports and Parameters

In [1]:
import pandas as pd

from src.member3_utils import (
    load_scenarios,
    plot_customer_scatter,
    plot_metric_bar,
    plot_routes,
    run_scenario_workflow,
    validate_scenario,
)

NUM_RIDERS = 5
RIDER_CAPACITY = 10
RANDOM_SEED = 6106

## 2. Load and Validate Data

In [2]:
scenarios = load_scenarios()
validation = pd.DataFrame([validate_scenario(df, name) for name, df in scenarios.items()])
validation

,scenario,rows,city_codes,dates,min_order_hour,max_order_hour,missing_key_values
0,Jaipur,50,JAP,02-04-2022,17.0,23.0,0
1,Mumbai,50,MUM,02-03-2022,17.0,23.0,0
2,Hyderabad,50,HYD,18-03-2022,17.0,23.0,0


## 3. Run Baselines and Optimized Heuristic

In [3]:
scenario_outputs = {
    name: run_scenario_workflow(df, name)
    for name, df in scenarios.items()
}

all_results = pd.concat(
    [scenario_result.results for scenario_result in scenario_outputs.values()],
    ignore_index=True,
)

all_results.round(3)

,scenario,method,total_distance_km,avg_distance_per_rider_km,max_distance_per_rider_km,min_distance_per_rider_km,workload_imbalance_km,max_orders_per_rider,min_orders_per_rider,estimated_total_time_min,improvement_vs_original_pct
0,Jaipur,Original Order,472.537,94.507,133.281,73.942,59.339,10,10,1692.977,0.000
1,Jaipur,Random Assignment,514.278,102.856,118.404,92.197,26.207,10,10,1849.544,-8.833
2,Jaipur,Geographic Clustering,319.475,63.895,88.419,51.022,37.397,20,3,1093.101,32.392
3,Jaipur,Geographic + Nearest Neighbor,224.868,44.974,49.208,40.192,9.016,20,3,810.889,52.413
4,Jaipur,Balanced Geo + NN + 2-opt,247.324,49.465,55.790,44.946,10.844,10,10,871.263,47.661
5,Mumbai,Original Order,768.190,153.638,182.894,131.818,51.076,10,10,2557.678,0.000
6,Mumbai,Random Assignment,746.935,149.387,184.128,109.167,74.961,10,10,2634.713,2.767
7,Mumbai,Geographic Clustering,416.846,83.369,126.916,59.657,67.259,16,4,1435.937,45.737
8,Mumbai,Geographic + Nearest Neighbor,287.846,57.569,74.717,38.992,35.725,16,4,982.288,62.529
9,Mumbai,Balanced Geo + NN + 2-opt,293.432,58.686,78.919,36.843,42.076,10,10,975.267,61.802


## 4. Jaipur Route Visualization

In [4]:
jaipur = scenarios["Jaipur"]
jaipur_routes = scenario_outputs["Jaipur"].routes

plot_customer_scatter(jaipur, "Jaipur Customer Locations and Common Depot")
plot_routes(jaipur, jaipur_routes["Original Order"], "Jaipur Original Order Baseline Routes")
plot_routes(jaipur, jaipur_routes["Balanced Geo + NN + 2-opt"], "Jaipur Optimized Heuristic Routes")

c:\Users\Zhang\OneDrive - Nanyang Technological University\桌面\NTU学习\6106-ADVANCED PRESCRIPTIVE ANALYTICS WITH GENERATIVE AI\6106_Project\src\member3_utils.py:486: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Metric Comparison

In [5]:
plot_metric_bar(all_results, "total_distance_km", "Total Distance by Scenario and Method")
plot_metric_bar(all_results, "workload_imbalance_km", "Workload Imbalance by Scenario and Method")

c:\Users\Zhang\OneDrive - Nanyang Technological University\桌面\NTU学习\6106-ADVANCED PRESCRIPTIVE ANALYTICS WITH GENERATIVE AI\6106_Project\src\member3_utils.py:486: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
c:\Users\Zhang\OneDrive - Nanyang Technological University\桌面\NTU学习\6106-ADVANCED PRESCRIPTIVE ANALYTICS WITH GENERATIVE AI\6106_Project\src\member3_utils.py:486: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Export Results

In [6]:
from pathlib import Path

Path("outputs/tables").mkdir(parents=True, exist_ok=True)
validation.to_csv("outputs/tables/member3_validation_summary.csv", index=False)
all_results.round(3).to_csv("outputs/tables/member3_results_summary.csv", index=False)